In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import LineCollection
from scipy.ndimage import gaussian_filter1d
from matplotlib import gridspec
#import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from dataclasses import dataclass, replace
# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

work_path = "M:/1confiProj/"
os.chdir(work_path)
from scripts.modelClassGPU import ConfiModel
from scripts.experiments import Experiment, DataHandler
from scripts.evaluators import ModelEvaluator
from scripts.plot_function import *
#from utils.help_function import *

device is : cpu


In [4]:
# save condition - adhoc
import pickle

# #get conditions from the model class
# experiment = Experiment('base_clamp')
# model = ConfiModel(m_id = 1, exp_config = experiment.exp_config)
# conds = model.conditions

# with open('P:/3026008.02/fit_res/model_condition.pkl', 'wb') as f:
#     pickle.dump(conds, f)

with open('P:/3026008.02/fit_res/model_condition.pkl', 'rb') as f:
    conds = pickle.load(f)

In [2]:
# define the model
exp_name = 'fixed_bug'
#exp = Experiment(exp_name=exp_name, cluster=False)
#dh = DataHandler(cluster=False)
ev = ModelEvaluator(exp_name=exp_name, cluster=False)
m_id,sub_id = 2,2
model = ev.get_model(m_id=m_id)

In [17]:
# reference parameters for the model
df_ref = ev.get_best_param_nll(m_id=m_id, sub_id=3, by='train')
df_ref

(array([2.97838804e+00, 8.16740215e+00, 3.32562562e+00, 9.70512564e+00,
        6.03990950e+00, 1.27044423e+01, 2.31297520e-03, 5.91259465e-02,
        9.57334477e+00, 2.06892749e+01, 6.29914033e-01, 5.09092242e-01,
        8.17325845e-02, 7.97575310e-02]),
 np.float64(22760.63325871096))

In [4]:
ev.dh.get_fit_param_names(m_id=m_id, exp_name=exp_name)

['sig_Vrh',
 'sig_Vrl',
 'sig_A',
 'kC1',
 'mC1',
 'a',
 'b',
 'w_vis',
 'sig_P_h',
 'sig_P_l',
 'p_com_h',
 'p_com_l',
 'rngA',
 'rngV']

In [7]:
param_bounds = dict(zip(model.estParamsNames, model.bounds))
param_bounds

{'sig_Vrh': (0.001, 5),
 'sig_Vrl': (0.1, 20),
 'sig_A': (0.1, 20),
 'kC1': (1e-08, 50),
 'mC1': (-10, 10),
 'a': (1e-08, 50),
 'b': (1e-08, 10),
 'w_vis': (0.01, 1),
 'sig_P_h': (1, 60),
 'sig_P_l': (1, 60),
 'p_com_h': (0.0001, 0.9999),
 'p_com_l': (0.0001, 0.9999),
 'rngA': (1e-08, 0.8),
 'rngV': (1e-08, 0.8)}

In [8]:
fixed_params = [
        # 'sig_Vrh',
        # 'sig_Vrl',
        # 'sig_A',
        # 'kC1',
        # 'mC1',
        # 'w_vis',
        # 'wb_v',
        # 'wb_a',
        # 'sig_P',
        # 'p_com',
        # 'rng',
        #'sig_rs',
        #'sig_rconf',
        #'sig_rc'
        ]

tune_params = [p for p in model.estParamsNames if p not in fixed_params]

In [9]:
tune_params

['sig_Vrh',
 'sig_Vrl',
 'sig_A',
 'kC1',
 'mC1',
 'a',
 'b',
 'w_vis',
 'sig_P_h',
 'sig_P_l',
 'p_com_h',
 'p_com_l',
 'rngA',
 'rngV']

In [ ]:
sliders = {}
for p in tune_params:
    p_min, p_max = param_bounds[p]
    sliders[p] = FloatSlider(min=p_min, max=p_max, step=(p_max - p_min)/100, value=(p_min+p_max)/2, description=p)

In [11]:
sliders 

{'sig_Vrh': FloatSlider(value=2.5005, description='sig_Vrh', max=5.0, min=0.001, step=0.04999),
 'sig_Vrl': FloatSlider(value=10.05, description='sig_Vrl', max=20.0, min=0.1, step=0.19899999999999998),
 'sig_A': FloatSlider(value=10.05, description='sig_A', max=20.0, min=0.1, step=0.19899999999999998),
 'kC1': FloatSlider(value=25.000000005, description='kC1', max=50.0, min=1e-08, step=0.4999999999),
 'mC1': FloatSlider(value=0.0, description='mC1', max=10.0, min=-10.0, step=0.2),
 'a': FloatSlider(value=25.000000005, description='a', max=50.0, min=1e-08, step=0.4999999999),
 'b': FloatSlider(value=5.000000005, description='b', max=10.0, min=1e-08, step=0.0999999999),
 'sig_P_h': FloatSlider(value=30.5, description='sig_P_h', max=60.0, min=1.0, step=0.59),
 'sig_P_l': FloatSlider(value=30.5, description='sig_P_l', max=60.0, min=1.0, step=0.59),
 'p_com_h': FloatSlider(value=0.5, description='p_com_h', max=0.9999, min=0.0001, step=0.009998),
 'p_com_l': FloatSlider(value=0.5, descriptio

In [14]:
@dataclass
class ModelDefaultParams:
    """Stores numerical model parameters with defaults."""
    sig_Vrh: float = 1
    sig_Vrl: float = 7
    sig_V: float = 5
    sig_A: float = 8
    sig_P: float = 30
    sig_P_h: float = 30
    sig_P_l: float = 30
    sig_rs: float = 0.01
    sig_rconf: float = 0.01
    sig_rc: float = 0.01
    gamma_rate: float = 2
    mu_P: float = 0
    p_com_h: float = 0.5
    p_com_l: float = 0.5
    rng: float = 0.3
    rngA: float = 0.3
    rngV: float = 0.3
    logbeta: float = 0
    logbeta_conf: float = 0
    a:float = 1
    b:float = 0
    kC1:float = 5
    mC1:float = 1
    kC2:float = 5
    mC2:float = 1
    #w_vis: float = 0 
    wb_v: float = 0
    wb_a: float = 0
    w_vis: float = 0
    w_vis_h: float = 0
    w_vis_l: float = 0

In [12]:
def plot_confidence_interval(df):
    df = df.loc[df.Modality == 'A']

    df_plot = df.groupby(['sub_id', 'VisRelLabel', 'abs_delta_VA', 'AudLoc']).CISize.mean().reset_index()
    palette = sns.color_palette("viridis", as_cmap=False)

    fig, axs = plt.subplots(figsize = (10, 5), nrows=1, ncols=2, sharey=True)
    sns.lineplot(
        data=df_plot[df_plot.VisRelLabel == 'High'],
        x='AudLoc',
        y='CISize',
        hue='abs_delta_VA',
        palette=palette,
        linewidth=2.5,
        ax = axs[0],
        legend = None,
        estimator = 'mean',
        errorbar='se',
        err_style='bars',
    )
    sns.lineplot(
        data=df_plot[df_plot.VisRelLabel == 'Low'],
        x='AudLoc',
        y='CISize',
        hue='abs_delta_VA',
        palette=palette,
        linewidth=2.5,
        ax = axs[1],
        estimator = 'mean',
        errorbar='se',
        err_style='bars',
        #legend = None,
    )
    clean_axs(axs[0])
    clean_axs(axs[1])

    axs[1].legend(
        title='Absolute spatial disparity ($^\\circ$)',
        frameon=False,
        loc='lower center',
        ncol=5,
        bbox_to_anchor=(0, -0.5),
        fontsize=fontsize-2,
        title_fontsize=fontsize-2
    )
    axs[0].set_title('High visual reliability', fontsize=fontsize)
    axs[1].set_title('Low visual reliability', fontsize=fontsize)
    axs[0].set_ylabel('Confidence interval', fontsize=fontsize)
    axs[0].set_xlabel('Auditory location ($^\\circ$)', fontsize=fontsize)
    axs[1].set_xlabel('Auditory location ($^\\circ$)', fontsize=fontsize)
    plt.tight_layout()
    plt.show()


In [13]:
def plot_causal_conf(df_sim):
    _, axs = plt.subplots(figsize=(10, 5), ncols=3, nrows=1, sharey=True,
                            gridspec_kw={'wspace': 0.2, 'hspace': 0.4})
    plot_group_pattern(df_sim, 'ConfLevel', 'Common', axs[0], trial_type='A')
    plot_group_pattern(df_sim, 'ConfLevel', 'Total', axs[1], trial_type='A')
    plot_group_pattern(df_sim, 'ConfLevel', 'Separate', axs[2], trial_type='A')


In [15]:
def plot_simulation_conf(**params):
    # params is a dict: {param_name: value}
    default_params = ModelDefaultParams()
    model_params = replace(default_params, **params)
    inputs = [getattr(model_params, p) for p in model.estParamsNames]
    trialConds = model.getCondRep(500)  # simulate 500 trials
    dat = model.run_simulation(inputs, trialConds=trialConds)
    df_sim = ev.dh.organize_dat(dat = dat)
    #plot the simulation results
    plot_confidence_interval(df_sim)

interact(plot_simulation_conf, **sliders)

interactive(children=(FloatSlider(value=2.5004999999999997, description='sig_Vrh', max=5.0, min=0.001, step=0.…

<function __main__.plot_simulation_conf(**params)>

In [ ]:
def plot_simulation_causal(**params):
    # params is a dict: {param_name: value}
    default_params = ModelDefaultParams()
    model_params = replace(default_params, **params)
    inputs = [getattr(model_params, p) for p in model.estParamsNames]
    trialConds = model.getCondRep(500)  # simulate 500 trials
    dat = model.run_simulation(inputs, trialConds=trialConds)
    df_sim = ev.dh.organize_dat(dat = dat)
    #plot the simulation results
    plot_causal_conf(df_sim)

interact(plot_simulation_causal, **sliders)

interactive(children=(FloatSlider(value=2.5005, description='sig_Vrh', max=5.0, min=0.001, step=0.04999), Floa…

<function __main__.plot_simulation_causal(**params)>